# aw_11_seeds — Gate G6: 3-seed final confirmation (protocol §8 + Amendment v1.4)

**Scope (pre-registered in Amendment v1.4):** reseed the *final-stage Phase-2
training* of champion **B4v2** and Track-A control **A2v2** with seeds **43, 44**
(s42 = existing runs-of-record), on identical sha-pinned parents and the identical
frozen data artifact. Full frozen-suite eval per run; greedy decoding.

**Headline requirement:** sign consistency of per-suite B4v2−A2v2 pass-rate deltas
across all 3 seeds; report mean ± sd (single-seed labels removed from final tables
only for the two arms covered here).

Cell order: `a_seeds_setup` → `b_seeds_train`(×4) → `c_seeds_eval`(×4) → `x20_gate`
→ `f_seeds_analysis` → `g_seed_aggregate`.

**Stop rules (§10):** any diverging run is marked `failed` and reported — no
hyperparameter retries. **No new development configs are allowed at this stage.**

**Monitoring per training run (first ~50 steps):** loss finite and decreasing;
eval-JSON validity not collapsing; for SFT, terminal `<|im_end|>` behavior healthy
(v2 adapter contract, modules_to_save=[lm_head, embed_tokens] inherited from the
resolved configs — do NOT re-derive configs by hand).


In [ ]:
# @title common header
import os, sys

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt
sys.path.append("src")
!python scripts/audit_runtime.py


Cloning into 'axiom-world'...
remote: Enumerating objects: 806, done.
remote: Counting objects: 100% (355/355), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 806 (delta 214), reused 229 (delta 102), pack-reused 451 (from 1)
Receiving objects: 100% (806/806), 353.26 KiB | 5.05 MiB/s, done.
Resolving deltas: 100% (446/446), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 221.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 

In [ ]:
# @title a_seeds_setup — runs-of-record + parents + frozen inputs (v2.1, protocol-v1 safe)
import subprocess
import json
import pathlib

from axiom_world.core.fingerprints import fingerprint_payload


# ---- runs of record (s42) -------------------------------------------------
B4V2_REPO = "m97j/aw-runs-b4"
B4V2_RUN  = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"

A2V2_REPO = "m97j/aw-runs-a2"
A2V2_RUN  = "20260814-114257--a2v2-playworld-dpo--s42--ef1e20"
A2V2_EVAL = "20260814-120733--eval-playworld--s42--61d7cd"

SEEDS = [43, 44]  # s42 already exists as run-of-record


def sh(*args):
    print("+", " ".join(args))
    r = subprocess.run(list(args))
    assert r.returncode == 0, f"FAILED: {args}"


def load_json(path):
    return json.loads(pathlib.Path(path).read_text())


def lineage(run):
    return load_json(f"runs/{run}/artifacts/lineage.json")


def fingerprint_jsonl(path):
    """
    Protocol-v1 dataset fingerprint verification.

    IMPORTANT:
    This intentionally uses the same semantic fingerprinting scheme
    used by the existing lineage records, rather than raw file-byte SHA256.
    """
    path = pathlib.Path(path)

    records = []
    with path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise AssertionError(
                    f"Invalid JSONL at {path}:{lineno}: {exc}"
                ) from exc

    return len(records), fingerprint_payload(records)


def fetch_and_verify_dataset(
    *,
    repo,
    repo_path,
    output,
    expected_fingerprint,
):
    """
    Fetch persisted dataset bytes using the existing v1 fetch CLI,
    then verify them using the protocol's dataset fingerprint semantics.

    fetch_dataset.py's byte SHA is intentionally NOT compared to the
    lineage dataset_fingerprint because lineage uses fingerprint_payload().
    """

    sh(
        "python",
        "scripts/fetch_dataset.py",
        "--repo",
        repo,
        "--path",
        repo_path,
        "--output",
        output,
        "--force",
    )

    records, actual_fingerprint = fingerprint_jsonl(output)

    print(f"records: {records}")
    print(f"dataset fingerprint: {actual_fingerprint}")
    print(f"expected fingerprint: {expected_fingerprint}")

    assert (
        actual_fingerprint == expected_fingerprint
    ), (
        "DATASET FINGERPRINT MISMATCH\n"
        f"  path:     {output}\n"
        f"  expected: {expected_fingerprint}\n"
        f"  actual:   {actual_fingerprint}\n"
    )

    print("dataset fingerprint verified:", output)
    return actual_fingerprint


# ---------------------------------------------------------------------------
# 1) Fetch the two runs-of-record
# ---------------------------------------------------------------------------

for repo, run in [
    (B4V2_REPO, B4V2_RUN),
    (A2V2_REPO, A2V2_RUN),
]:
    sh(
        "python",
        "scripts/fetch_run.py",
        "--repo",
        repo,
        "--run-id",
        run,
    )


lin_b4 = lineage(B4V2_RUN)
lin_a2 = lineage(A2V2_RUN)

print(
    json.dumps(
        {"b4v2": lin_b4, "a2v2": lin_a2},
        indent=2,
    )[:3000]
)


# ---------------------------------------------------------------------------
# 2) Resolve and fetch parents
# ---------------------------------------------------------------------------

def parent_pin(lin):
    rid = lin.get("parent_run_id")
    parent = lin.get("parent_adapter") or {}

    repo = parent.get("repo_id")
    sha = parent.get("sha256")

    assert rid and repo, (
        f"could not resolve parent pin from lineage: {lin.keys()}"
    )

    return rid, repo, sha


P1_RUN, P1_REPO, P1_SHA = parent_pin(lin_b4)
A1V2_RUN, A1V2_REPO, A1V2_SHA = parent_pin(lin_a2)

sh(
    "python",
    "scripts/fetch_run.py",
    "--repo",
    P1_REPO,
    "--run-id",
    P1_RUN,
)

sh(
    "python",
    "scripts/fetch_run.py",
    "--repo",
    A1V2_REPO,
    "--run-id",
    A1V2_RUN,
)


# Cross-check fetched parent adapter against child's lineage pin.
from axiom_world.core.lineage import compute_adapter_sha256

for run, pin in [
    (P1_RUN, P1_SHA),
    (A1V2_RUN, A1V2_SHA),
]:
    actual = (
        "sha256:"
        + compute_adapter_sha256(
            pathlib.Path(f"runs/{run}/artifacts/final_adapter")
        ).removeprefix("sha256:")
    )

    assert pin is None or actual == pin, (
        f"parent adapter sha mismatch for {run}: "
        f"{actual} != {pin}"
    )

    print("parent pin verified:", run)


P1_PARENT_DIR = (
    f"runs/{P1_RUN}/artifacts/final_adapter"
)

A1V2_PARENT_DIR = (
    f"runs/{A1V2_RUN}/artifacts/final_adapter"
)

print("P1_PARENT_DIR =", P1_PARENT_DIR)
print("A1V2_PARENT_DIR =", A1V2_PARENT_DIR)


# ---------------------------------------------------------------------------
# 3) Frozen SFT training data
#
# Existing lineage fingerprint is fingerprint_payload(records),
# NOT raw-file SHA256.
# ---------------------------------------------------------------------------

B4_SFT_FP = lin_b4["dataset_fingerprints"]["sft"]

fetch_and_verify_dataset(
    repo="m97j/aw-playworld",
    repo_path="train/v1/playworld_sft.jsonl",
    output="data/train/playworld_sft.jsonl",
    expected_fingerprint=B4_SFT_FP,
)

A2_PREF_FP = lin_a2["dataset_fingerprints"]["preference"]

fetch_and_verify_dataset(
    repo="m97j/aw-playworld",
    repo_path="preference_train/a1v2-v1/playworld_preference_a1.jsonl",
    output="data/train/playworld_preference_a1.jsonl",
    expected_fingerprint=A2_PREF_FP,
)


+ python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id 20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2
+ python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id 20260814-114257--a2v2-playworld-dpo--s42--ef1e20
{
  "b4v2": {
    "base_model_repo_id": "Qwen/Qwen3-8B-Base",
    "base_model_revision": "49e3418fbbbca6ecbdf9608b4d22e5a407081db4",
    "code_commit": "ff984e4bef63a2a04d3606b01e01da4152a91a21",
    "config_fingerprint": "sha256:65a28426257d8e4e7b608903530dd0006217c2115e6531e990cf48ba65ecb462",
    "dataset_fingerprints": {
      "sft": "sha256:54fcb1d364e49b7e3468dffe0030c37fdbf3f27a33cd83b124f9c7bd3f32afb1"
    },
    "initialization_mode": "continue_training_existing_adapter",
    "output_adapter_sha256": "sha256:d4fcacddf21f758cdab904845ebdfee1eefde309c0edb6205bac64d5f07c76c8",
    "parent_adapter": {
      "repo_id": "m97j/aw-runs-b1",
      "revision": "main",
      "sha256": "sha256:747e57574f811274d34852f8347c47c2ff90a86186f616f796856c7f01cae21d"
    

'sha256:5856b5c5e2fc8fa9b20d1611431c13197ebbf780937020e73cf41064424167a9'

In [ ]:
# @title b_seeds_train — B4v2 + A2v2 reseeds (4 runs, sequential)
# Replicate from each run-of-record's resolved_config.yaml, overriding ONLY the
# seed and the experiment name. Parents come from a_seeds_setup (hash-verified
# local dirs materialized by fetch_run — NOT hub paths).
import subprocess

JOBS = []
DONE = {("b4v2-playworld-sft-from-p1-s43", 43)}
for seed in SEEDS:
    JOBS.append(dict(
        config=f"runs/{B4V2_RUN}/artifacts/resolved_config.yaml",
        name=f"b4v2-playworld-sft-from-p1-s{seed}",
        parent=P1_PARENT_DIR, repo="m97j/aw-runs-seeds", seed=seed,
    ))
    JOBS.append(dict(
        config=f"runs/{A2V2_RUN}/artifacts/resolved_config.yaml",
        name=f"a2v2-playworld-dpo-s{seed}",
        parent=A1V2_PARENT_DIR, repo="m97j/aw-runs-seeds", seed=seed,
    ))

for j in JOBS:
    if (j["name"], j["seed"]) in DONE:
        print("skip (already completed):", j["name"]); continue
    r = subprocess.run([
        "python", "scripts/run_experiment.py",
        "--config", j["config"],
        "--override", f"runtime.seed={j['seed']}",
        "--override", f"experiment_name={j['name']}",
        "--parent-adapter-dir", j["parent"],
        "--hf-sync-repo", j["repo"],
    ])
    assert r.returncode == 0, f"train failed: {j['name']} — STOP, do not retune (protocol §10)"


skip (already completed): b4v2-playworld-sft-from-p1-s43


In [ ]:
# @title restore_missing_b4v2_s43 — revision-pinned adapter restore
from pathlib import Path
import json

from huggingface_hub import snapshot_download

from axiom_world.core.lineage import compute_adapter_sha256

B4V2_S43_REPO = "m97j/aw-runs-seeds"
B4V2_S43_RUN = "20260817-151254--b4v2-playworld-sft-from-p1-s43--s43--c04b9c"
B4V2_S43_REVISION = "b49b5aa9cd94afa4df0f34304fee1daaad08331c"

RUN_ROOT = Path("runs") / B4V2_S43_RUN
ARTIFACTS = RUN_ROOT / "artifacts"
ADAPTER = ARTIFACTS / "final_adapter"
LINEAGE = ARTIFACTS / "lineage.json"

# 1. Revision-pinned materialization.
snapshot_download(
    repo_id=B4V2_S43_REPO,
    repo_type="model",
    revision=B4V2_S43_REVISION,
    allow_patterns=["artifacts/*"],
    local_dir=str(RUN_ROOT),
)

# 2. Basic existence checks.
assert ADAPTER.is_dir(), f"missing adapter: {ADAPTER}"
assert LINEAGE.is_file(), f"missing lineage: {LINEAGE}"

# 3. Verify that this revision actually corresponds to the requested run.
lineage = json.loads(LINEAGE.read_text())

assert lineage["run_id"] == B4V2_S43_RUN, (
    "HF revision does not contain the requested run.\n"
    f"expected: {B4V2_S43_RUN}\n"
    f"actual:   {lineage.get('run_id')}"
)

# 4. Verify adapter bytes against the persisted lineage hash.
expected = lineage["output_adapter_sha256"]
actual = compute_adapter_sha256(ADAPTER)

assert actual == expected, (
    "ADAPTER INTEGRITY FAILURE\n"
    f"expected: {expected}\n"
    f"actual:   {actual}"
)

print("restored:", B4V2_S43_RUN)
print("revision:", B4V2_S43_REVISION)
print("adapter:", ADAPTER)
print("adapter sha256:", actual)
print("lineage hash verified: OK")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

restored: 20260817-151254--b4v2-playworld-sft-from-p1-s43--s43--c04b9c
revision: b49b5aa9cd94afa4df0f34304fee1daaad08331c
adapter: runs/20260817-151254--b4v2-playworld-sft-from-p1-s43--s43--c04b9c/artifacts/final_adapter
adapter sha256: sha256:4dd1dd33c577811074d6704e2d0d1b7dccc112a8af043ee4d6a06e5e1d390758
lineage hash verified: OK


In [ ]:
# @title c_seeds_eval — frozen-suite eval for all 4 new adapters
import subprocess
import glob

!python scripts/build_eval_suites.py --episodes-per-suite 300

ADAPTERS = {}
for j in JOBS:
    cand = sorted(glob.glob(f"runs/*--{j['name']}--s{j['seed']}--*/artifacts/final_adapter"))
    assert cand, f"no adapter for {j['name']}"
    ADAPTERS[(j['name'], j['seed'])] = cand[-1]

EVAL_RUNS = {}
for key, adapter in ADAPTERS.items():
    r = subprocess.run([
        "python", "scripts/run_evaluation.py",
        "--config", "configs/experiments/eval_playworld.yaml",
        "--adapter-dir", adapter,
        "--hf-sync-repo", "m97j/aw-runs-seeds",
    ])
    assert r.returncode == 0, f"eval failed: {key}"
    EVAL_RUNS[key] = sorted(glob.glob("runs/*--eval-playworld--*"))[-1]
print(EVAL_RUNS)


eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).
{('b4v2-playworld-sft-from-p1-s43', 43): 'runs/20260818-100258--eval-playworld--s42--304567', ('a2v2-playworld-dpo-s43', 43): 'runs/20260818-101023--eval-playworld--s42--44a500', ('b4v2-playworld-sft-from-p1-s44', 44): 'runs/20260818-105059--eval-playworld--s42--082b2a', ('a2v2-playworld-dpo-s44', 44): 'runs/20260818-105846--eval-playworld-

In [ ]:
# @title x20_gate — eval identity audit (mandatory after the stale-weights incident)
# Every pair of eval runs that should differ MUST NOT be prediction-identical.
import subprocess
import itertools

runs = [f"{r}/artifacts" for r in EVAL_RUNS.values()] + [
    f"runs/{B4V2_EVAL}/artifacts"
]

for i, (a, b) in enumerate(itertools.combinations(runs, 2)):
    out = f"runs/x20_identity_audit_{i:02d}.json"

    r = subprocess.run([
        "python", "scripts/x20_eval_identity_audit.py",
        "--run-a", a,
        "--run-b", b,
        "--out", out,
    ], capture_output=True, text=True)

    print(r.stdout)

    assert r.returncode == 0, (
        f"identity audit FAILED: {a} vs {b}"
    )

print("x20 identity audits: PASS")

{
  "run_a": "runs/20260818-100258--eval-playworld--s42--304567/artifacts",
  "run_b": "runs/20260818-101023--eval-playworld--s42--44a500/artifacts",
  "episodes_paired": 1500,
  "identical_predictions": 0,
  "identical_rate": 0.0,
  "verdict": "DIFFERENT_WEIGHTS",
  "verdict_meaning": "evals are genuinely distinct adapters",
  "per_suite": {
    "eval_adversarial": {
      "episodes": 300,
      "identical": 0,
      "identical_rate": 0.0
    },
    "eval_comp_ood": {
      "episodes": 300,
      "identical": 0,
      "identical_rate": 0.0
    },
    "eval_id": {
      "episodes": 300,
      "identical": 0,
      "identical_rate": 0.0
    },
    "eval_rule_ood": {
      "episodes": 300,
      "identical": 0,
      "identical_rate": 0.0
    },
    "eval_template_ood": {
      "episodes": 300,
      "identical": 0,
      "identical_rate": 0.0
    }
  },
  "sample_diffs": [
    {
      "suite": "eval_adversarial",
      "episode": "eval_adversarial-eval_adversarial-fam0-0000",
      "a_h

In [ ]:
# @title f0_rebuild_eval_runs — rediscover the 4 seed eval runs from hub (no re-eval)
import json, pathlib, re
from huggingface_hub import HfApi

SEEDS_REPO = "m97j/aw-runs-seeds"
api = HfApi()

# 1) discover eval run ids on the hub
eval_ids = sorted({
    m.group(1)
    for f in api.list_repo_files(SEEDS_REPO)
    if (m := re.match(r"runs/(\d{8}-\d{6}--eval-playworld--s\d+--[0-9a-f]+)/", f))
})
print("found eval runs on hub:", *eval_ids, sep="\n  ")
assert len(eval_ids) == 4, f"expected 4 seed eval runs, found {len(eval_ids)}"

# 2) fetch each and map back to (train_name, seed) via its summary's adapter_dir
def summary_of(run_dir):
    for cand in (pathlib.Path(run_dir, "artifacts", "evaluation_summary.json"),
                 pathlib.Path(run_dir, "evaluation_summary.json")):
        if cand.is_file():
            return json.loads(cand.read_text())
    raise FileNotFoundError(run_dir)

EVAL_RUNS = {}
for ev in eval_ids:
    sh("python", "scripts/fetch_run.py", "--repo", SEEDS_REPO, "--run-id", ev, "--kind", "eval")
    s = summary_of(f"runs/{ev}")
    ad = s["adapter_dir"]  # e.g. runs/20260817-...--b4v2-playworld-sft-from-p1-s43--s43--c04b9c/...
    m = re.search(r"--((?:b4v2-playworld-sft-from-p1|a2v2-playworld-dpo)-s(\d+))--s\d+--", ad)
    assert m, f"cannot parse adapter_dir: {ad}"
    name, seed = m.group(1), int(m.group(2))
    assert (name, seed) not in EVAL_RUNS, f"duplicate eval for {name}"
    EVAL_RUNS[(name, seed)] = f"runs/{ev}"
    print(f"  {name} (s{seed}) -> {ev}  id_pass={s['suites']['eval_id']['pass_rate']['mean']:.4f}")

assert len(EVAL_RUNS) == 4

# 3) fetch runs of record (s42)
!python scripts/fetch_run.py --repo {B4V2_REPO} --run-id {B4V2_EVAL} --kind eval

found eval runs on hub:
  20260818-100258--eval-playworld--s42--304567
  20260818-101023--eval-playworld--s42--44a500
  20260818-105059--eval-playworld--s42--082b2a
  20260818-105846--eval-playworld--s42--40b6f2
+ python scripts/fetch_run.py --repo m97j/aw-runs-seeds --run-id 20260818-100258--eval-playworld--s42--304567 --kind eval
  b4v2-playworld-sft-from-p1-s43 (s43) -> 20260818-100258--eval-playworld--s42--304567  id_pass=0.4067
+ python scripts/fetch_run.py --repo m97j/aw-runs-seeds --run-id 20260818-101023--eval-playworld--s42--44a500 --kind eval
  a2v2-playworld-dpo-s43 (s43) -> 20260818-101023--eval-playworld--s42--44a500  id_pass=0.1833
+ python scripts/fetch_run.py --repo m97j/aw-runs-seeds --run-id 20260818-105059--eval-playworld--s42--082b2a --kind eval
  b4v2-playworld-sft-from-p1-s44 (s44) -> 20260818-105059--eval-playworld--s42--082b2a  id_pass=0.3800
+ python scripts/fetch_run.py --repo m97j/aw-runs-seeds --run-id 20260818-105846--eval-playworld--s42--40b6f2 --kind eval

In [ ]:
# @title f_seeds_analysis — per-seed paired B4v2 vs A2v2 + champion stability
import subprocess
for seed in SEEDS:
    a = EVAL_RUNS[(f"b4v2-playworld-sft-from-p1-s{seed}", seed)]
    b = EVAL_RUNS[(f"a2v2-playworld-dpo-s{seed}", seed)]
    subprocess.run(["python", "scripts/run_analysis.py",
        "--run-a", a, "--label-a", f"b4v2-s{seed}",
        "--run-b", b, "--label-b", f"a2v2-s{seed}",
        "--output", f"{a}/analysis_b4v2_vs_a2v2_s{seed}.json",
        "--hf-sync-repo", "m97j/aw-runs-seeds"], check=True)
# seed-to-seed drift of the champion itself (s43/s44 vs s42 run-of-record):
for seed in SEEDS:
    a = EVAL_RUNS[(f"b4v2-playworld-sft-from-p1-s{seed}", seed)]
    subprocess.run(["python", "scripts/run_analysis.py",
        "--run-a", a, "--label-a", f"b4v2-s{seed}",
        "--run-b", f"runs/{B4V2_EVAL}", "--label-b", "b4v2-s42",
        "--output", f"{a}/analysis_b4v2_s{seed}_vs_s42.json",
        "--hf-sync-repo", "m97j/aw-runs-seeds"], check=True)


In [ ]:
# @title g_seed_aggregate — x21 (v2: baselines explicitly fetched + hard-gated)
import subprocess, json, pathlib

# 1) BOTH s42 baseline eval runs must be materialized (B4V2_EVAL was never fetched before!)
for repo, ev in [("m97j/aw-runs-b4", B4V2_EVAL), ("m97j/aw-runs-a2", A2V2_EVAL)]:
    sh("python", "scripts/fetch_run.py", "--repo", repo, "--run-id", ev, "--kind", "eval")

entries = [("b4v2", f"runs/{B4V2_EVAL}", 42),
           ("a2v2", f"runs/{A2V2_EVAL}", 42)]
for (name, seed), run in EVAL_RUNS.items():
    entries.append(("b4v2" if name.startswith("b4v2") else "a2v2", run, seed))

# 2) HARD GATE: every entry's summary must exist at ITS OWN path — no fallback allowed.
#    Print identifying fields so a wrong file is visible before aggregation.
def summary_of(run_dir):
    for cand in (pathlib.Path(run_dir, "artifacts", "evaluation_summary.json"),
                 pathlib.Path(run_dir, "evaluation_summary.json")):
        if cand.is_file():
            return json.loads(cand.read_text())
    raise FileNotFoundError(f"no evaluation_summary.json under {run_dir} — REFUSING to aggregate")

for model, run, seed in entries:
    s = summary_of(run)
    print(f"{model} s{seed}: adapter={s.get('adapter_dir','?')[:70]} "
          f"id_pass={s['suites']['eval_id']['pass_rate']['mean']:.4f} "
          f"freeze={s['freeze_fingerprint'][:24]}")
    assert s["freeze_fingerprint"].startswith("sha256:3cdcbc30"), f"suite fingerprint mismatch: {run}"

# sanity: the two b4v2/a2v2 s42 baselines must NOT be identical rows
assert summary_of(f"runs/{B4V2_EVAL}")["suites"]["eval_id"]["pass_rate"]["mean"] != \
       summary_of(f"runs/{A2V2_EVAL}")["suites"]["eval_id"]["pass_rate"]["mean"]

# 3) aggregate
args = ["python", "scripts/x21_seed_variance.py", "--output", "runs/seed_variance_report.json"]
for model, run, seed in entries:
    args += ["--model", model, "--eval-run", run, "--seed", str(seed)]
subprocess.run(args, check=True)
print(json.load(open("runs/seed_variance_report.json"))["verdict"])


+ python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id 20260814-032546--eval-playworld--s42--7308ee --kind eval
+ python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id 20260814-120733--eval-playworld--s42--61d7cd --kind eval
b4v2 s42: adapter=runs/20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2/artifact id_pass=0.3933 freeze=sha256:3cdcbc30c99e492c4
a2v2 s42: adapter=runs/20260814-114257--a2v2-playworld-dpo--s42--ef1e20/artifacts/final_ id_pass=0.1867 freeze=sha256:3cdcbc30c99e492c4
b4v2 s43: adapter=runs/20260818-151254--b4v2-playworld-sft-from-p1-s43--s43--c04b9c/arti id_pass=0.4067 freeze=sha256:3cdcbc30c99e492c4
a2v2 s43: adapter=runs/20260818-085710--a2v2-playworld-dpo-s43--s43--0ae8d6/artifacts/fi id_pass=0.1833 freeze=sha256:3cdcbc30c99e492c4
b4v2 s44: adapter=runs/20260818-090731--b4v2-playworld-sft-from-p1-s44--s44--1c8f72/arti id_pass=0.3800 freeze=sha256:3cdcbc30c99e492c4
a2v2 s44: adapter=runs/20260818-092200--a2v2-playworld-dpo-s44--s44--453eaf/artifa

In [ ]:
import json

with open("runs/seed_variance_report.json", encoding="utf-8") as f:
    print(json.dumps(json.load(f)["verdict"], indent=2))

{
  "criterion": "sign(b4v2-a2v2) pass_rate identical across seeds",
  "per_suite": {
    "eval_id": {
      "delta_per_seed": {
        "42": 0.2066,
        "43": 0.2234,
        "44": 0.1933
      },
      "sign_consistent": true
    },
    "eval_template_ood": {
      "delta_per_seed": {
        "42": 0.14,
        "43": 0.15,
        "44": 0.1366
      },
      "sign_consistent": true
    },
    "eval_comp_ood": {
      "delta_per_seed": {
        "42": 0.16,
        "43": 0.2033,
        "44": 0.2066
      },
      "sign_consistent": true
    },
    "eval_rule_ood": {
      "delta_per_seed": {
        "42": 0.1,
        "43": 0.1166,
        "44": 0.1134
      },
      "sign_consistent": true
    },
    "eval_adversarial": {
      "delta_per_seed": {
        "42": 0.21,
        "43": 0.2033,
        "44": 0.1867
      },
      "sign_consistent": true
    }
  },
  "pass": true,
  "seeds": [
    42,
    43,
    44
  ]
}


## Deliverables to bring back to the assistant after this notebook
1. `runs/seed_variance_report.json` (full JSON)
2. The four `analysis_*.json` outputs of `f_seeds_analysis`
3. run_ids + adapter sha256 of the 4 new runs (from run_card.json)
4. Any `failed` run's event log tail if a stop rule fired
5. x20 gate output (PASS lines)

If the sign-consistency verdict is PASS → proceed to §7 write-up (tech report).
If any suite flips sign across seeds → the report's headline is weakened to the
suites that remain consistent; do NOT rerun with new seeds (that would be
seed-shopping and violates §10).
